# 🚀 Phase 3: The Transformer Bosst (BERT Fine-Tuning)

## 🏗 Overview: State-of-the-Art NLP
In the previous phases, we proved that **Semantics (Word2Vec)** outperform **Frequencies (TF-IDF)** with a 98.2% accuracy. Now, we enter the final boss level: **Transformers**.

We will fine-tune a **DistilBERT** model. DistilBERT uses an **Attention Mechanism** to understand the bidirectional context of every word in an article. While Word2Vec treats words as fixed vectors, Transformers understand that a word's meaning changes based on all other words in the sentence.

## 1. Setup & Environment (Colab GPU Recommended)
Fine-tuning a Transformer is computationally expensive. We recommend running this notebook in **Google Colab** with a **T4 GPU**.

In [ ]:
import os
import sys

# 1. Environment Detection
if 'google.colab' in sys.modules:
    !pip install -q transformers[torch] datasets evaluate accelerate umap-learn
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
    print(f"✅ Colab GPU Environment. BASE_PATH: {BASE_PATH}")
else:
    BASE_PATH = '.'
    print("💻 Local Environment. Note: Training will be slow without GPU.")

import pandas as pd
import numpy as np
import torch
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

## 2. Data Loading & Preparation
We load our cleaned dataset and convert it into the HuggingFace `Dataset` format.

In [ ]:
# Load cleaned data (from Phase 1)
df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/cleaned_data.csv'))
df['full_text'] = df['full_text'].fillna('')

# We only need 'full_text' and 'label'
# Note: We take a sample if we want to test the pipeline quickly (e.g., sample=None for full training)
dataset_full = Dataset.from_pandas(df[['full_text', 'label']])
dataset_split = dataset_full.train_test_split(test_size=0.2, seed=42)

print(f"✅ Dataset loaded. Training size: {len(dataset_split['train'])}, Test size: {len(dataset_split['test'])}")

## 3. Tokenization (DistilBERT)
Transformers don't use simple regex cleaning; they use **Subword Tokenization** (WordPiece). This handles out-of-vocabulary words better than Word2Vec.

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Truncate to 512 tokens (BERT limit)
    return tokenizer(examples["full_text"], padding="max_length", truncation=True)

tokenized_datasets = dataset_split.map(tokenize_function, batched=True)
print("✅ Tokenization complete.")

## 4. Fine-Tuning Management
We use the HuggingFace `Trainer` API to fine-tune our model for 2 or 3 epochs.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir=os.path.join(BASE_PATH, "models/bert_results"),
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=1,
    report_to="none"
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

### 5. Train and Save the new model as distilbert_classifier

In [ ]:

# 🚀 INICIAR ENTRENAMIENTO (Necesitas GPU en Colab)
print("⏳ Iniciando entrenamiento del modelo DistilBERT...")
trainer.train()

# 💾 GUARDAR MODELO FINAL PARA GRÁFICAS Y COMPARACIÓN
final_model_path = os.path.join(BASE_PATH, 'models/distilbert_classifier')
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"✅ Entrenamiento completado. Modelo guardado en: {final_model_path}")

## 7. Final Delivery: Predictions on Validation Data
We use our best model to generate the final competition file.

In [ ]:
def generate_predictions(val_csv_path, output_csv_path):
    val_df = pd.read_csv(val_csv_path)
    
    # 1. Preprocess validation text (Title + Text)
    # ⚠️ Prevención ante NaNs: rellenamos con texto vacío antes de concatenar
    val_df['full_text'] = val_df['title'].fillna('') + " " + val_df['text'].fillna('')
    
    # Note: For BERT, we use raw text to allow BERT to capture punctuation and context.
    val_dataset = Dataset.from_pandas(val_df[['full_text']])
    tokenized_val = val_dataset.map(tokenize_function, batched=True)
    
    # 2. Predict (Trainer handles GPU batching)
    raw_preds = trainer.predict(tokenized_val)
    predictions = np.argmax(raw_preds.predictions, axis=-1)
    
    # 3. Replace labels with predictions
    val_df['label'] = predictions
    
    # 4. Filter and Export safely
    ideal_columns = ['label', 'title', 'text', 'subject', 'date']
    final_columns = [col for col in ideal_columns if col in val_df.columns]
    
    val_df[final_columns].to_csv(output_csv_path, index=False, sep=',')
    print(f"\ud83c? Final Deliverable generated at: {output_csv_path}")

valid_path = os.path.join(BASE_PATH, 'dataset/validation_data.csv')
final_path = os.path.join(BASE_PATH, 'dataset/final_validation_results_bert.csv')



In [ ]:
# FINAL EXECUTION: Generate the deliverable for the competition
# -------------------------------------------------------------------------------------


if os.path.exists(valid_path):

    generate_predictions(valid_path, final_path)
    print(f" Final deliverable is ready at: {final_path}")
else:
    print(f"Validation file not found at {valid_path}. Check your dataset folder!")